# Transformer for Jet Classification

This task uses a different dataset. Each datapoint is 20 separate particles (a subset of the particles which make up the event).

In [ ]:
import h5py
import numpy as np
from numpy.lib.recfunctions import structured_to_unstructured

In [ ]:
import torch
print(torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

True
Using cuda device


In [ ]:
import os
import urllib.request
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

filename = "/content/drive/MyDrive/part_transformer.h5"
url = "https://cernbox.cern.ch/s/4rtGlo1RlFvUNwz/download/"

if not os.path.exists(filename):
    print(f"Downloading {filename}...")
    urllib.request.urlretrieve(url, filename)
    print("Download complete.")
else:
    print(f"{filename} already exists. Skipping download.")

In [10]:
with h5py.File("part_transformer.h5","r") as f:
  signal_data = f["signal"][:]
  bkg_data = f["bkg"][:]

OSError: Unable to synchronously open file (truncated file: eof = 20975186, sblock->base_addr = 0, stored_eof = 349254496)

In [ ]:
signal = structured_to_unstructured(signal_data)
background = structured_to_unstructured(bkg_data)

# Task starts here
* Concatenate the signal and background features into one array, and assign the appropriate target labels
* Split into training and testing datasets
* Scale the data ot some senisble scale
* Build a model which has 1) an MLP to embed the 5 features in some larger space 2) some transformer encoder layers 3) a pooling layer to condense the information in 1D format 4) a final MLP for the final classification
* Proceed with training (I only used the training dataset and used the test dataset solely for comparison at the end, with no validation dataset).

## Important
My transformer had 2 encoder layers with 2 attention heads, a batch size of 4096, a learning rate of 0.001, and trained for 20 epochs. It took about one minute per epoch to train. Making your transformer larger than this will increase the training time.

In [ ]:
X = np.concatenate([signal,background])
X.shape #Already in the transformer shape we need i.e. Ndata x Ntokens x Nfeatures

In [ ]:
target = np.concatenate([np.ones(signal.shape[0]), np.zeros(background.shape[0])], axis=0)


In [ ]:
from sklearn.model_selection import train_test_split

# First split: 80% train, 20% temp
X_train, X_temp, Y_train, Y_temp = train_test_split(X, target, test_size=0.2, random_state=42)

# Second split: 10% validation, 10% test
X_val, X_test, Y_val, Y_test = train_test_split(X_temp, Y_temp, test_size=0.5, random_state=42)

print(f"Train: {X_train.shape[0]} samples")
print(f"Validation: {X_val.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train.reshape(-1, 5))  # 5 features per jet

X_train_scaled = scaler.transform(X_train.reshape(-1, 5)).reshape(X_train.shape)
X_val_scaled = scaler.transform(X_val.reshape(-1, 5)).reshape(X_val.shape)
X_test_scaled = scaler.transform(X_test.reshape(-1, 5)).reshape(X_test.shape)

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32, device=device)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32, device=device)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32, device=device)

Y_train_tensor = torch.tensor(Y_train.reshape(-1, 1), dtype=torch.float32, device=device)
Y_val_tensor = torch.tensor(Y_val.reshape(-1, 1), dtype=torch.float32, device=device)
Y_test_tensor = torch.tensor(Y_test.reshape(-1, 1), dtype=torch.float32, device=device)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class JetTransformer(nn.Module):

    """
    Transformer model for jet classification
    First step is an embedder for particles
    Next step is a set of transformer layers
    Last step is a classifier head
    """

    def __init__(self, input_dim, nhead, num_layers):
        super(JetTransformer, self).__init__()

        # Input embedding MLP
        self.input_embedder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
        )
        # Transformer Encoder - made of TransformerEncoderLayers
        self.transformer_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=32, nhead=nhead,batch_first=True),
            num_layers=num_layers, enable_nested_tensor=False)

        # Output classifier MLP
        self.output_classifier_head = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1))

    def mean_pooling(self,x):
        """
        Mean pooling implementation
        """
        # x: (batch_size, num_tokens, embed_dim)
        return x.mean(dim=1)


    def forward(self, x):
        x = self.input_embedder(x)
        x = self.transformer_encoder(x)
        x = self.mean_pooling(x)
        x = self.output_classifier_head(x)
        return torch.sigmoid(x)

In [ ]:
model = JetTransformer(input_dim=5, nhead=4, num_layers=2)
model = model.to(device)

In [ ]:

from torch.utils.data import TensorDataset, DataLoader

batch_size = 8192

train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, Y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
# Define optimiser and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()


In [ ]:

print("\nBeginning training loop")
print("=" * 60)

train_losses = []
val_losses = []

num_epochs = 30

for epoch in range(num_epochs):
    start_time = time.time()  # Start timer

    # --- Training ---
    model.train()
    running_train_loss = 0.0
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- Validation ---
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            running_val_loss += loss.item()

    avg_val_loss = running_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    epoch_time = time.time() - start_time  # End timer

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Time: {epoch_time:.2f}s")

    torch.cuda.empty_cache()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Evaluate by looping over the test_dataset
model.eval()
list_of_predictions = []
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)  # if using GPU

        outputs = model(inputs)
        list_of_predictions.append(outputs)
        loss = criterion(outputs, targets)

In [ ]:
pred = torch.concatenate(list_of_predictions)
Y_pred = torch.round(pred).detach().cpu().numpy()

from sklearn.metrics import roc_auc_score
print(roc_auc_score(Y_test.reshape(-1,1),Y_pred))